ValueError: Input size (3) must equal the number of input parameters generated by the circuit (0).

QLayer(
  (layer): QuantumLayer(
    (_photon_loss_transform): PhotonLossTransform()
    (_detector_transform): DetectorTransform()
    (measurement_mapping): Probabilities()
  )
  (second_interf): QuantumLayer(
    (_photon_loss_transform): PhotonLossTransform()
    (_detector_transform): DetectorTransform()
    (measurement_mapping): Probabilities()
  )
  (scaler): ScaleLayer()
)

[0, 1, 2, 3, 4, 5, 6]

In [5]:
import torch
import torch.nn as nn
from merlin.builder import CircuitBuilder
from merlin import QuantumLayer


class ScaleLayer(nn.Module):
    def __init__(self, dim, scale_type="learned"):
        super(ScaleLayer, self).__init__()
        if scale_type == "learned":
            self.scale = nn.Parameter(torch.rand(dim))
        elif scale_type == "2pi":
            self.register_buffer('scale', torch.full((dim,), 2 * torch.pi))
        elif scale_type == "pi":
            self.register_buffer('scale', torch.full((dim,), torch.pi))
        elif scale_type == "1":
            self.register_buffer('scale', torch.full((dim,), 1.0))

    def forward(self, x):
        return x * self.scale


class QLayer(nn.Module):
    def __init__(self, nb_photons, nb_modes, dim, scale_type="learned"):
        super().__init__() 
        self.nb_modes = nb_modes
        self.nb_photons = nb_photons

        

        builder = CircuitBuilder(nb_modes)


        builder.add_entangling_layer(name="Generic interferometer BS+PS", trainable=True)
        builder.add_angle_encoding(list(range(nb_modes)))

        builder.add_rotations(modes=list(range(nb_modes)), trainable=True)


        self.layer = QuantumLayer(input_size=nb_modes,
                                  builder=builder,
                                  n_photons=nb_photons, 
                                  dtype=torch.float32)
        
        
        
        
        builder_second_interf = CircuitBuilder(nb_modes)

        
        builder_second_interf.add_angle_encoding(list(range(nb_modes)))
        builder_second_interf.add_rotations(list(range(nb_modes)))
        

        self.second_interf = QuantumLayer(input_size=nb_modes,
                                          builder=builder_second_interf,
                                          n_photons=nb_photons,
                                          dtype=torch.float32)
        self.scaler = ScaleLayer(dim, scale_type)


    # 4. CORRECTION SYNTAXE : L'indentation de forward()

    def forward(self, x):
        # 5. CORRECTION TYPO : self.layer(x) au lieu de self.second_inter(x)
        x = self.layer(x)
        x = self.scaler(x)
        x = self.second_interf(x)
        return x

QLayer(3,3,3) 

QLayer(
  (layer): QuantumLayer(
    (_photon_loss_transform): PhotonLossTransform()
    (_detector_transform): DetectorTransform()
    (measurement_mapping): Probabilities()
  )
  (second_interf): QuantumLayer(
    (_photon_loss_transform): PhotonLossTransform()
    (_detector_transform): DetectorTransform()
    (measurement_mapping): Probabilities()
  )
  (scaler): ScaleLayer()
)